# Lab | Data Structuring and Combining Data

## Challenge 1: Combining & Cleaning Data

In this challenge, we will be working with the customer data from an insurance company, as we did in the two previous labs. The data can be found here:
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv

But this time, we got new data, which can be found in the following 2 CSV files located at the links below.

- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv
- https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv

Note that you'll need to clean and format the new data.

Observation:
- One option is to first combine the three datasets and then apply the cleaning function to the new combined dataset
- Another option would be to read the clean file you saved in the previous lab, and just clean the two new files and concatenate the three clean datasets

In [21]:
%%writefile cleaning_functions.py
#import the data:
import pandas as pd  

def main_cleaning(df):
    df = df.copy()
    
    # standarize and clean the columns 
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = pd.to_numeric(
            df['customer_lifetime_value'].astype(str).str.replace('%', ''), 
            errors='coerce'
        ) 
    df = df.dropna().ffill().bfill()
    
    # delete duplicates
    if 'customer' in df.columns:
        df = df.drop_duplicates(subset='customer', keep='first')
    else:
        df = df.drop_duplicates()
    
    # convert to enter numbers
    num_cols = df.select_dtypes('number').columns
    df[num_cols] = df[num_cols].astype(int)
    
    return df.reset_index(drop=True)
import pandas as pd
import importlib
import cleaning_functions

# Apply changes of %%writefile
importlib.reload(cleaning_functions)
from cleaning_functions import main_cleaning

# URLs of the data
urls = [
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file1.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file2.csv",
    "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/file3.csv"
]

# concatenate the data
df_raw = pd.concat([pd.read_csv(url) for url in urls], ignore_index=True)

# clean
df_final = main_cleaning(df_raw)

print(f"final result: {df_final.shape[0]} .")
df_final.head()

Overwriting cleaning_functions.py


# Challenge 2: Structuring Data

In this challenge, we will continue to work with customer data from an insurance company, but we will use a dataset with more columns, called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by performing data cleaning, formatting, and structuring.

In [22]:
%%writefile cleaning_functions.py

#import the data
import pandas as pd

def main_cleaning(df):
    # standarize and clean the columns 
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = pd.to_numeric(
            df['customer_lifetime_value'].astype(str).str.replace('%', ''), 
            errors='coerce'
        )
    
    df = df.dropna().reset_index(drop=True)
    
    # delete duplicates of customer id
    if 'customer' in df.columns:
        df = df.drop_duplicates(subset='customer', keep='first')
    
    # 5. reset numerics
    num_cols = df.select_dtypes('number').columns
    df[num_cols] = df[num_cols].astype(int)
    
    return df.reset_index(drop=True)
    import pandas as pd
import importlib
import cleaning_functions

# Reload 
importlib.reload(cleaning_functions)
from cleaning_functions import main_cleaning
url_extended = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df_marketing = pd.read_csv(url_extended)
df_marketing_clean = main_cleaning(df_marketing)

print(f"Dataset: {df_marketing_clean.shape[0]} filas y {df_marketing_clean.shape[1]} columns.")
display(df_marketing_clean.head())

Overwriting cleaning_functions.py


1. You work at the marketing department and you want to know which sales channel brought the most sales in terms of total revenue. Using pivot, create a summary table showing the total revenue for each sales channel (branch, call center, web, and mail).
Round the total revenue to 2 decimal points.  Analyze the resulting table to draw insights.

In [23]:
%%writefile cleaning_functions.py
import pandas as pd

def main_cleaning(df):
    #Copy and standarized columns 
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = pd.to_numeric(
            df['customer_lifetime_value'].astype(str).str.replace('%', '', regex=False), 
            errors='coerce'
        )
    df = df.dropna().reset_index(drop=True)
    if 'customer' in df.columns:
        df = df.drop_duplicates(subset='customer', keep='first')
    num_cols = df.select_dtypes('number').columns
    df[num_cols] = df[num_cols].astype(int)
    
    return df.reset_index(drop=True)
    import pandas as pd
import importlib
import cleaning_functions

# Reload to clean the previous one
importlib.reload(cleaning_functions)
from cleaning_functions import main_cleaning

# download the data
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df_raw = pd.read_csv(url)

# clean
df_marketing_clean = main_cleaning(df_raw)

# create the pivot
sales_channel_revenue = df_marketing_clean.pivot_table(
    index='sales_channel', 
    values='customer_lifetime_value', 
    aggfunc='sum'
).round(2).sort_values(by='customer_lifetime_value', ascending=False)

display(sales_channel_revenue)

Overwriting cleaning_functions.py


2. Create a pivot table that shows the average customer lifetime value per gender and education level. Analyze the resulting table to draw insights.

In [16]:
%%writefile cleaning_functions.py
import pandas as pd

def main_cleaning(df):
    # create a copy
    df = df.copy()
    
    # standarize and clean the columns
    df.columns = df.columns.str.lower().str.replace(' ', '_')
    if 'customer_lifetime_value' in df.columns:
        df['customer_lifetime_value'] = pd.to_numeric(
            df['customer_lifetime_value'].astype(str).str.replace('%', '', regex=False), 
            errors='coerce'
        )
    df = df.dropna().reset_index(drop=True)
    if 'customer' in df.columns:
        df = df.drop_duplicates(subset='customer')
    num_cols = df.select_dtypes('number').columns
    df[num_cols] = df[num_cols].fillna(0).astype(int)
    
    return df.reset_index(drop=True)
import pandas as pd
import importlib
import cleaning_functions

# Forzamos la carga del archivo limpio
importlib.reload(cleaning_functions)
from cleaning_functions import main_cleaning

# 1. Cargar el dataset
url_extended = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df_raw = pd.read_csv(url_extended)

# 2. Limpiar (Aquí creamos df_marketing_clean)
df_marketing_clean = main_cleaning(df_raw)

# 3. CREAR EL PIVOT DE GÉNERO Y EDUCACIÓN
pivot_gender_edu = df_marketing_clean.pivot_table(
    index='education', 
    columns='gender', 
    values='customer_lifetime_value', 
    aggfunc='mean'
).round(2)

print("--- Tabla: Media de CLV por Educación y Género ---")
display(pivot_gender_edu)

Overwriting cleaning_functions.py


## Bonus

You work at the customer service department and you want to know which months had the highest number of complaints by policy type category. Create a summary table showing the number of complaints by policy type and month.
Show it in a long format table.

*In data analysis, a long format table is a way of structuring data in which each observation or measurement is stored in a separate row of the table. The key characteristic of a long format table is that each column represents a single variable, and each row represents a single observation of that variable.*

*More information about long and wide format tables here: https://www.statology.org/long-vs-wide-data/*

In [18]:
import pandas as pd
from cleaning_functions import main_cleaning

# 1. Cargar el dataset directamente
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis_clean.csv"
df_raw = pd.read_csv(url)

# 2. Limpiar los datos para asegurar que las columnas existan
df_marketing_clean = main_cleaning(df_raw)

# 3. Crear la tabla en LONG FORMAT
# Agrupamos por mes y tipo de póliza y contamos los clientes
complaints_long = df_marketing_clean.groupby(['month', 'policy_type'])['customer'].count().reset_index()

# 4. Renombrar la columna de resultados
complaints_long.columns = ['month', 'policy_type', 'number_of_complaints']

# 5. Ordenar para ver los meses con más quejas primero
complaints_long = complaints_long.sort_values(by=['month', 'number_of_complaints'], ascending=[True, False])

# Mostrar el resultado final
print("Summary Table: Complaints by Month and Policy Type (Long Format)")
display(complaints_long)

--- Tabla: Media de CLV por Educación y Género ---


gender,F,M
education,,
Bachelor,8012.39,7721.56
College,7732.32,7970.83
Doctor,7423.12,7614.35
High School or Below,8578.50,8009.58
Master,8318.59,8157.57


--- Tabla: Media de CLV por Educación y Género ---


gender,F,M
education,,
Bachelor,8012.39,7721.56
College,7732.32,7970.83
Doctor,7423.12,7614.35
High School or Below,8578.50,8009.58
Master,8318.59,8157.57


Summary Table: Complaints by Month and Policy Type (Long Format)


,month,policy_type,number_of_complaints
1,1,Personal Auto,3641
0,1,Corporate Auto,1051
2,1,Special Auto,206
4,2,Personal Auto,3147
3,2,Corporate Auto,917
5,2,Special Auto,172
